In [2]:
# Step 1.1: Config
NESSIE_URL = "http://nessie:19120/api/v1"
BASE_BRANCH = "dev"
TARGET_BRANCH = "gold_dev"
SPARK_MASTER = "spark://spark:7077"
WAREHOUSE = "s3a://football-lake/"


In [3]:
# Step 1.2: Ensure Nessie branch exists (idempotent)
import requests
refs = requests.get(f"{NESSIE_URL}/trees", timeout=30).json().get("references", [])
main_ref = next((r for r in refs if r.get("type") == "BRANCH" and r.get("name") == BASE_BRANCH), None)
if main_ref is None:
    raise RuntimeError(f"Base branch {BASE_BRANCH!r} not found in Nessie.")
target_ref = next((r for r in refs if r.get("type") == "BRANCH" and r.get("name") == TARGET_BRANCH), None)
if target_ref is None:
    payload = {"type": "BRANCH", "name": TARGET_BRANCH, "hash": main_ref["hash"]}
    r = requests.post(f"{NESSIE_URL}/trees/tree", params={"sourceRefName": BASE_BRANCH}, json=payload, timeout=30)
    if r.status_code not in (200, 201, 409):
        raise RuntimeError(f"Failed creating branch: {r.status_code} {r.text}")
refs_after = requests.get(f"{NESSIE_URL}/trees", timeout=30).json().get("references", [])
main_after = next(r for r in refs_after if r.get("type") == "BRANCH" and r.get("name") == BASE_BRANCH)
target_after = next(r for r in refs_after if r.get("type") == "BRANCH" and r.get("name") == TARGET_BRANCH)
print("main hash  :", main_after["hash"])
print("target hash:", target_after["hash"])
print("branch_ready:", target_after["name"] == TARGET_BRANCH)


main hash  : 16befa0a01101b61895f5c3aa82f8f807cd8e3e78cec308ed995ab1568792228
target hash: 16befa0a01101b61895f5c3aa82f8f807cd8e3e78cec308ed995ab1568792228
branch_ready: True


In [ ]:
# Step 1.3: Spark init (robust jar path detection)
import glob, os, sys
from pyspark.sql import SparkSession
for p in ["/opt/spark/python", "/usr/local/spark/python"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
jar_dirs = ["/opt/spark/jars-extra", "/usr/local/spark/jars-extra"]
jar_dir = next((d for d in jar_dirs if os.path.isdir(d)), None)
if jar_dir is None:
    raise RuntimeError(f"No jars-extra directory found. Checked: {jar_dirs}")
jars = sorted(glob.glob(f"{jar_dir}/*.jar"))
if not jars:
    raise RuntimeError(f"No jars found in {jar_dir}. Check docker-compose volume mount.")
try:
    spark.stop()
except Exception:
    pass
spark = (SparkSession.builder
    .appName("gold_sql_notebook")
    .master(SPARK_MASTER)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", NESSIE_URL)
    .config("spark.sql.catalog.nessie.ref", TARGET_BRANCH)
    .config("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.region", "us-east-1")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password")
    .config("spark.sql.defaultCatalog", "nessie")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.jars", ",".join(jars))
    .config("spark.driver.extraClassPath", f"{jar_dir}/*")
    .config("spark.executor.extraClassPath", f"{jar_dir}/*")
    .getOrCreate())
print("spark_catalog_ref:", spark.conf.get("spark.sql.catalog.nessie.ref"))
print("jar_dir:", jar_dir)
print("jar_count:", len(jars))


26/04/28 14:46:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


spark_catalog_ref: gold_dev
jar_dir: /opt/spark/jars-extra
jar_count: 5


In [ ]:
silver_df= spark.table("nessie.silver.player_team_match_stats")


silver_df.show(5)
silver_df.printSchema()
silver_df.describe().show()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+---------+-----------------+-------+----------------+---------+----------+--------------+-------------------+-----------+----------+----------+----------+------------+-----------------+--------------------+--------------------+
|match_id|player_id|      player_name|team_id|       team_name|season_id|match_week|competition_id|             source|event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|                  xg|     ingested_at_utc|
+--------+---------+-----------------+-------+----------------+---------+----------+--------------+-------------------+-----------+----------+----------+----------+------------+-----------------+--------------------+--------------------+
| 3895266|     3500|     Granit Xhaka|    904|Bayer Leverkusen|      281|        10|             9|statsbomb_open_data|        494|       167|         2|         0|           0|                3|         0.043321265|2026-04-27 19:10:...|
| 3895266|     7044|    Patrik Schick|    904|Ba

26/04/28 14:47:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+---------------+-----------------+---------+---------+------------------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|summary|          match_id|         player_id|    player_name|          team_id|team_name|season_id|        match_week|competition_id|             source|       event_count|        pass_count|        shot_count|         goal_count|       assist_count| shot_assist_count|                 xg|
+-------+------------------+------------------+---------------+-----------------+---------+---------+------------------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|  count|               306|               306|            306|              306|      306|      306|               306|    

In [16]:
from pyspark.sql import functions as F

# build KPI dataframe at player-match grain
player_match_kpis_df = (
    silver_df
    .select(
        "season_id",
        "match_week",
        "competition_id",
        "match_id",
        "player_id",
        "player_name",
        "team_id",
        "team_name",
        "event_count",
        "pass_count",
        "shot_count",
        "goal_count",
        "assist_count",
        "shot_assist_count",
        "xg"
    )
    .groupBy(
        "season_id",
        "match_week",
        "competition_id",
        "match_id",
        "player_id",
        "player_name",
        "team_id",
        "team_name"
    )
    .agg(
        F.sum("event_count").alias("event_count"),
        F.sum("pass_count").alias("pass_count"),
        F.sum("shot_count").alias("shot_count"),
        F.sum("goal_count").alias("goal_count"),
        F.sum("assist_count").alias("assist_count"),
        F.sum("shot_assist_count").alias("shot_assist_count"),
        F.sum("xg").alias("xg")
    )
    # contribution metrics
    .withColumn("goal_contribution", F.col("goal_count") + F.col("assist_count"))
    .withColumn(
        "involvement_index",
        F.col("goal_count") * F.lit(4)
        + F.col("assist_count") * F.lit(3)
        + F.col("shot_count")
        + F.col("shot_assist_count")
    )
    # efficiency metrics 
    .withColumn(
        "xg_per_shot",
        F.when(F.col("shot_count") > 0, F.col("xg") / F.col("shot_count")).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "shot_conversion",
        F.when(F.col("shot_count") > 0, F.col("goal_count") / F.col("shot_count")).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "shot_assist_ratio",
        F.when(F.col("pass_count") > 0, F.col("shot_assist_count") / F.col("pass_count")).otherwise(F.lit(None).cast("double"))
    )
)


In [17]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.gold.player_match_kpis (
  season_id INT,
  match_week INT,
  competition_id INT,
  match_id INT,
  player_id INT,
  player_name STRING,
  team_id INT,
  team_name STRING,
  event_count INT,
  pass_count INT,
  shot_count INT,
  goal_count INT,
  assist_count INT,
  shot_assist_count INT,
  xg DOUBLE,
  goal_contribution INT,
  involvement_index INT,
  xg_per_shot DOUBLE,
  shot_conversion DOUBLE,
  shot_assist_ratio DOUBLE
)
USING iceberg
PARTITIONED BY (season_id, match_week)
""")

player_match_kpis_df.writeTo("nessie.gold.player_match_kpis").overwritePartitions()

spark.sql("""
SELECT season_id, match_week, COUNT(*) AS rows
FROM nessie.gold.player_match_kpis
GROUP BY season_id, match_week
ORDER BY season_id, match_week
""").show(100, truncate=False)

spark.table("nessie.gold.player_match_kpis").show(5, truncate=False)
spark.table("nessie.gold.player_match_kpis").printSchema()

+---------+----------+----+
|season_id|match_week|rows|
+---------+----------+----+
|281      |5         |30  |
|281      |8         |29  |
|281      |10        |31  |
|281      |11        |31  |
|281      |13        |30  |
|281      |14        |30  |
|281      |15        |30  |
|281      |18        |32  |
|281      |19        |31  |
|281      |20        |32  |
+---------+----------+----+

+---------+----------+--------------+--------+---------+--------------+-------+----------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+------------+---------------+--------------------+
|season_id|match_week|competition_id|match_id|player_id|player_name   |team_id|team_name       |event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|xg         |goal_contribution|involvement_index|xg_per_shot |shot_conversion|shot_assist_ratio   |
+---------+----------+--------------+--------+---------+---------

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# build KPI dataframe at player-season grain
player_season_kpis_df = (
    player_match_kpis_df
    .groupBy("season_id", "player_id")
    .agg(
        F.first("player_name", ignorenulls=True).alias("player_name"),
        F.first("team_id", ignorenulls=True).alias("team_id"),
        F.first("team_name", ignorenulls=True).alias("team_name"),
        F.sum("event_count").alias("event_count"),
        F.sum("pass_count").alias("pass_count"),
        F.sum("shot_count").alias("shot_count"),
        F.sum("goal_count").alias("goal_count"),
        F.sum("assist_count").alias("assist_count"),
        F.sum("shot_assist_count").alias("shot_assist_count"),
        F.sum("xg").alias("xg"),
        F.sum("goal_contribution").alias("goal_contribution"),
        F.sum("involvement_index").alias("involvement_index"),
        F.countDistinct("match_id").alias("matches_played")
    )
    #weighted season metrics
    .withColumn(
        "xg_per_shot",
        F.when(F.col("shot_count") > 0, F.col("xg") / F.col("shot_count")).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "shot_conversion",
        F.when(F.col("shot_count") > 0, F.col("goal_count") / F.col("shot_count")).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "shot_assist_ratio",
        F.when(F.col("pass_count") > 0, F.col("shot_assist_count") / F.col("pass_count")).otherwise(F.lit(None).cast("double"))
    )
)

In [13]:
w_goal = Window.partitionBy("season_id").orderBy(F.col("goal_contribution").desc())
w_eff  = Window.partitionBy("season_id").orderBy(F.col("xg_per_shot").desc_nulls_last())
w_conv = Window.partitionBy("season_id").orderBy(F.col("shot_conversion").desc_nulls_last())
w_asr  = Window.partitionBy("season_id").orderBy(F.col("shot_assist_ratio").desc_nulls_last())

player_season_kpis_df = (
    player_season_kpis_df
    .withColumn("goal_contribution_rank", F.dense_rank().over(w_goal))
    .withColumn("efficiency_rank", F.dense_rank().over(w_eff))
    .withColumn("conversion_rank", F.dense_rank().over(w_conv))
    .withColumn("assist_ratio_rank", F.dense_rank().over(w_asr))
)


In [14]:
player_season_kpis_df.show(10, truncate=False)

+---------+---------+-----------------+-------+-------------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------+-----------+---------------+-------------------+----------------------+---------------+---------------+-----------------+
|season_id|player_id|player_name      |team_id|team_name          |event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|xg         |goal_contribution|involvement_index|matches_played|xg_per_shot|shot_conversion|shot_assist_ratio  |goal_contribution_rank|efficiency_rank|conversion_rank|assist_ratio_rank|
+---------+---------+-----------------+-------+-------------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------+-----------+---------------+-------------------+----------------------+---------------+---------------+-----------------+
|281      |8391 

In [15]:

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")


spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.gold.player_season_kpis (
  season_id INT,
  player_id INT,
  player_name STRING,
  team_id INT,
  team_name STRING,
  event_count BIGINT,
  pass_count BIGINT,
  shot_count BIGINT,
  goal_count BIGINT,
  assist_count BIGINT,
  shot_assist_count BIGINT,
  xg DOUBLE,
  goal_contribution BIGINT,
  involvement_index BIGINT,
  matches_played BIGINT,
  xg_per_shot DOUBLE,
  shot_conversion DOUBLE,
  shot_assist_ratio DOUBLE,
  goal_contribution_rank INT,
  efficiency_rank INT,
  conversion_rank INT,
  assist_ratio_rank INT
)
USING iceberg
PARTITIONED BY (season_id)
""")

player_season_kpis_df.writeTo("nessie.gold.player_season_kpis").overwritePartitions()

spark.sql("""
SELECT season_id, COUNT(*) AS rows
FROM nessie.gold.player_season_kpis
GROUP BY season_id
ORDER BY season_id
""").show(truncate=False)

spark.table("nessie.gold.player_season_kpis").show(10, truncate=False)


+---------+----+
|season_id|rows|
+---------+----+
|281      |176 |
+---------+----+

+---------+---------+-----------------+-------+-------------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------+-----------+---------------+-------------------+----------------------+---------------+---------------+-----------------+
|season_id|player_id|player_name      |team_id|team_name          |event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|xg         |goal_contribution|involvement_index|matches_played|xg_per_shot|shot_conversion|shot_assist_ratio  |goal_contribution_rank|efficiency_rank|conversion_rank|assist_ratio_rank|
+---------+---------+-----------------+-------+-------------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------+-----------+---------------+-------------------+----

In [18]:
player_match_kpis_df.show(2)

+---------+----------+--------------+--------+---------+-----------------+-------+---------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+-------------+---------------+-----------------+
|season_id|match_week|competition_id|match_id|player_id|      player_name|team_id|team_name|event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|         xg|goal_contribution|involvement_index|  xg_per_shot|shot_conversion|shot_assist_ratio|
+---------+----------+--------------+--------+---------+-----------------+-------+---------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+-------------+---------------+-----------------+
|      281|        20|             9| 3895348|     6874|Ermedin Demirović|    172| Augsburg|        118|        19|         4|         0|           0|                0|0.226136871|                0|       

In [19]:
match_kpis_df = spark.table("nessie.gold.player_match_kpis").select(
    "season_id",
    "match_week",
    "competition_id",
    "match_id",
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "goal_count",
    "assist_count",
    "shot_count",
    "xg",
    "goal_contribution"
)

w_last5 = (
    Window
    .partitionBy("season_id", "player_id")
    .orderBy(F.col("match_week").asc(), F.col("match_id").asc())
    .rowsBetween(-4, 0)
)


In [21]:
player_form_last5_df = (
    match_kpis_df
    .withColumn("matches_in_window", F.count(F.lit(1)).over(w_last5))
    .withColumn("goals_last5", F.sum("goal_count").over(w_last5))
    .withColumn("assists_last5", F.sum("assist_count").over(w_last5))
    .withColumn("shots_last5", F.sum("shot_count").over(w_last5))
    .withColumn("xg_last5", F.sum("xg").over(w_last5))
    .withColumn("goal_contribution_last5", F.sum("goal_contribution").over(w_last5))
    .withColumn("avg_xg_last5", F.avg("xg").over(w_last5))
    .withColumn(
        "shot_conversion_last5",
        F.when(F.col("shots_last5") > 0, F.col("goals_last5") / F.col("shots_last5"))
         .otherwise(F.lit(None).cast("double"))
    )

    .withColumn("first_xg_in_window", F.first("xg", ignorenulls=True).over(w_last5))
    .withColumn("last_xg_in_window", F.last("xg", ignorenulls=True).over(w_last5))
    .withColumn(
        "form_trend",
        F.when(F.col("last_xg_in_window") > F.col("first_xg_in_window"), F.lit("improving"))
         .when(F.col("last_xg_in_window") < F.col("first_xg_in_window"), F.lit("declining"))
         .otherwise(F.lit("stable"))
    )
    .drop("first_xg_in_window", "last_xg_in_window")
)

player_form_last5_df = player_form_last5_df.filter(F.col("matches_in_window") == 5)

In [22]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.gold.player_form_last5 (
  season_id INT,
  match_week INT,
  competition_id INT,
  match_id INT,
  player_id INT,
  player_name STRING,
  team_id INT,
  team_name STRING,
  goal_count INT,
  assist_count INT,
  shot_count INT,
  xg DOUBLE,
  goal_contribution INT,
  matches_in_window BIGINT,
  goals_last5 BIGINT,
  assists_last5 BIGINT,
  shots_last5 BIGINT,
  xg_last5 DOUBLE,
  goal_contribution_last5 BIGINT,
  avg_xg_last5 DOUBLE,
  shot_conversion_last5 DOUBLE,
  form_trend STRING
)
USING iceberg
PARTITIONED BY (season_id, match_week)
""")


player_form_last5_df.writeTo("nessie.gold.player_form_last5").overwritePartitions()

spark.sql("""
SELECT season_id, match_week, COUNT(*) AS rows
FROM nessie.gold.player_form_last5
GROUP BY season_id, match_week
ORDER BY season_id, match_week
""").show(200, truncate=False)

spark.table("nessie.gold.player_form_last5").show(10, truncate=False)
spark.table("nessie.gold.player_form_last5").printSchema()

+---------+----------+----+
|season_id|match_week|rows|
+---------+----------+----+
|281      |13        |9   |
|281      |14        |9   |
|281      |15        |13  |
|281      |18        |11  |
|281      |19        |13  |
|281      |20        |16  |
+---------+----------+----+

+---------+----------+--------------+--------+---------+---------------------------+-------+----------------+----------+------------+----------+-------------------+-----------------+-----------------+-----------+-------------+-----------+-------------------+-----------------------+--------------------+---------------------+----------+
|season_id|match_week|competition_id|match_id|player_id|player_name                |team_id|team_name       |goal_count|assist_count|shot_count|xg                 |goal_contribution|matches_in_window|goals_last5|assists_last5|shots_last5|xg_last5           |goal_contribution_last5|avg_xg_last5        |shot_conversion_last5|form_trend|
+---------+----------+--------------+--------

## Gold Data Quality 

In [23]:
from datetime import datetime

def run_gold_checks(df, table_name, key_cols, non_negative_cols, grain_cols):
    total = df.count()
    if total == 0:
        raise RuntimeError(f"{table_name}: hard check failed, no rows.")

    # Hard 1: null keys
    null_expr = " OR ".join([f"{c} IS NULL" for c in key_cols])
    null_keys = df.filter(null_expr).count()
    if null_keys > 0:
        raise RuntimeError(f"{table_name}: hard check failed, null keys={null_keys}")

    # Hard 2: impossible negatives
    neg_expr = " OR ".join([f"{c} < 0" for c in non_negative_cols])
    neg_rows = df.filter(neg_expr).count()
    if neg_rows > 0:
        raise RuntimeError(f"{table_name}: hard check failed, negative KPI rows={neg_rows}")

    # Hard 3: grain uniqueness
    dup_rows = (
        df.groupBy(*grain_cols).count().filter("count > 1").count()
    )
    if dup_rows > 0:
        raise RuntimeError(f"{table_name}: hard check failed, duplicate grain rows={dup_rows}")

    # Soft checks
    top10 = df.orderBy(F.col("goal_contribution").desc_nulls_last()).limit(10)
    print(f"[SOFT] Top 10 sanity sample for {table_name}")
    top10.show(truncate=False)

    return {
        "run_ts": datetime.utcnow().isoformat(),
        "table_name": table_name,
        "row_count": total,
        "null_key_rows": null_keys,
        "negative_rows": neg_rows,
        "duplicate_grain_rows": dup_rows,
    }

In [24]:
g1_metrics = run_gold_checks(
    player_match_kpis_df,
    "nessie.gold.player_match_kpis",
    key_cols=["season_id","match_week","match_id","player_id"],
    non_negative_cols=["event_count","pass_count","shot_count","goal_count","assist_count","shot_assist_count","xg","goal_contribution","involvement_index"],
    grain_cols=["season_id","match_week","match_id","player_id"]
)

g2_metrics = run_gold_checks(
    player_season_kpis_df,
    "nessie.gold.player_season_kpis",
    key_cols=["season_id","player_id"],
    non_negative_cols=["event_count","pass_count","shot_count","goal_count","assist_count","shot_assist_count","xg","goal_contribution","involvement_index","matches_played"],
    grain_cols=["season_id","player_id"]
)

g3_metrics = run_gold_checks(
    player_form_last5_df,
    "nessie.gold.player_form_last5",
    key_cols=["season_id","match_week","match_id","player_id"],
    non_negative_cols=["goal_count","assist_count","shot_count","xg","goals_last5","assists_last5","shots_last5","xg_last5","goal_contribution_last5"],
    grain_cols=["season_id","match_week","match_id","player_id"]
)

[SOFT] Top 10 sanity sample for nessie.gold.player_match_kpis
+---------+----------+--------------+--------+---------+------------------------------+-------+----------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------------+------------------+--------------------+
|season_id|match_week|competition_id|match_id|player_id|player_name                   |team_id|team_name       |event_count|pass_count|shot_count|goal_count|assist_count|shot_assist_count|xg         |goal_contribution|involvement_index|xg_per_shot         |shot_conversion   |shot_assist_ratio   |
+---------+----------+--------------+--------+---------+------------------------------+-------+----------------+-----------+----------+----------+----------+------------+-----------------+-----------+-----------------+-----------------+--------------------+------------------+--------------------+
|281      |15        |9             |3895302

[SOFT] Top 10 sanity sample for nessie.gold.player_form_last5
+---------+----------+--------------+--------+---------+----------------+-------+----------------+----------+------------+----------+--------------------+-----------------+-----------------+-----------+-------------+-----------+------------------+-----------------------+--------------------+---------------------+----------+
|season_id|match_week|competition_id|match_id|player_id|player_name     |team_id|team_name       |goal_count|assist_count|shot_count|xg                  |goal_contribution|matches_in_window|goals_last5|assists_last5|shots_last5|xg_last5          |goal_contribution_last5|avg_xg_last5        |shot_conversion_last5|form_trend|
+---------+----------+--------------+--------+---------+----------------+-------+----------------+----------+------------+----------+--------------------+-----------------+-----------------+-----------+-------------+-----------+------------------+-----------------------+---------------

## Audit Table

In [25]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold_audit")

spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.gold_audit.run_metrics (
  run_ts STRING,
  table_name STRING,
  row_count BIGINT,
  null_key_rows BIGINT,
  negative_rows BIGINT,
  duplicate_grain_rows BIGINT
) USING iceberg
""")

audit_rows = [g1_metrics, g2_metrics, g3_metrics]
audit_df = spark.createDataFrame(audit_rows)
audit_df.writeTo("nessie.gold_audit.run_metrics").append()